# Información del proyecto

## Propuesta inicial

En un contexto donde plataformas como Spotify o YouTube concentran gran parte del consumo musical, los artistas independientes enfrentan desafíos relacionados con la transparencia en los ingresos, la detección de usos no autorizados de su obra y la optimización de sus estrategias de lanzamiento. Este proyecto propone abordar estas problemáticas mediante el desarrollo de tres módulos principales basados en datos existentes. Los módulos son:

1. Análisis de mercado musical, orientado a identificar patrones de consumo, tendencias por género y oportunidades de monetización para artistas, permitiendo generar recomendaciones estratégicas basadas en datos (por ejemplo, mercados potenciales o características musicales asociadas al éxito).

2. Detección de fraude en streams, mediante técnicas de análisis de comportamiento y modelos de detección de anomalías, con el fin de identificar reproducciones artificiales o manipuladas que afectan la distribución justa de royalties.

3. Detector de copyright basado en inteligencia artificial, capaz de identificar la presencia de contenido musical en diferentes plataformas digitales mediante técnicas de procesamiento de audio, como espectrogramas y huellas digitales (audio fingerprinting), inspirado en tecnologías utilizadas por empresas como Shazam.

## Información de los módulos:



**1. Analisis del mercado musical apra el artista:**

* Datos posibles: Streams por país, Género musical, Playlist placement, TikTok trends, Crecimiento de artistas similares...

* Análisis (EDA): crecimiento por género, países con más crecimiento, correlación entre playlist y streams, horas/días con más reproducciones

* ML posible: Predicción de streams de una canción, Predicción de viralidad, 
* Modelos: Random Forest, XGBoost, LSTM para series temporales

* Output en tu plataforma

“Tu canción tiene 65% probabilidad de viralizarse en México”, “Este beat funciona mejor en playlist de chill trap”

* EXTRA DATOS: EDAD DE LA AUDIENCIA PARA AVERIGUAR TARGET SEGÚN PLATAFORMA. (MÉTRICAS AUDIENCIA TIK TOK)

**2. Fraude en streams:**

* Variables generales para objetivo 2:

| Variable                 | Tipo de dato | Para qué sirve                    | Ejemplo          |
| ------------------------ | ------------ | --------------------------------- | ---------------- |
| user_id                  | categorical  | identificar usuarios únicos       | U10233           |
| track_id                 | categorical  | identificar canción               | TRK8891          |
| session_id               | categorical  | identificar sesión                | S78123           |
| timestamp                | datetime     | detectar patrones temporales      | 2025-03-20 03:10 |
| listen_duration          | float        | detectar streams muy cortos       | 8.2 sec          |
| track_length             | float        | comparar duración real            | 180 sec          |
| completion_rate          | float        | ratio escucha completa            | 0.05             |
| device_type              | categorical  | bots usan dispositivos repetidos  | mobile / desktop |
| country                  | categorical  | detectar granjas de streams       | IN, RU           |
| playlist_source          | categorical  | detectar playlists sospechosas    | algorithmic      |
| repeat_count             | integer      | repeticiones consecutivas         | 25               |
| unique_tracks_in_session | integer      | bots suelen repetir pocas         | 1                |
| skip_rate                | float        | detectar skip masivo              | 0.90             |
| time_between_streams     | float        | bots tienen intervalos constantes | 5 sec            |


* Variables para futures engineering:

| Feature          | Fórmula                 |
| ---------------- | ----------------------- |
| repeat_ratio     | repeats / total streams |
| avg_listen_time  | mean(listen_duration)   |
| stream_frequency | streams / hour          |
| session_entropy  | diversidad de canciones |
| country_entropy  | diversidad geográfica   |


**3. Detector de copyright usando IA (Music Information Retrieval (MIR)):**

* Sistemas comerciales como Shazam usan huellas acústicas para identificar canciones.


* Modelo ML: con CNN usar como si fuera una imagen. 
* Al tratar info como imagen del sonido los ejes son: X = tiempo, Y = frecuencia y Color = energía.
* Variables generales para el punto 3: 

| Variable         | Tipo         | Para qué sirve                  | Ejemplo   |
| ---------------- | ------------ | ------------------------------- | --------- |
| track_id         | categorical  | identificar canción             | T123      |
| audio_file       | binary/audio | audio original                  | .wav      |
| sampling_rate    | int          | frecuencia audio                | 44100     |
| duration         | float        | duración track                  | 180s      |
| spectrogram      | matrix       | representación visual del audio | array     |
| mfcc_features    | vector       | características del timbre      | [12 dims] |
| chroma_features  | vector       | notas musicales                 | [12 dims] |
| tempo            | float        | BPM                             | 95        |
| pitch            | float        | tonalidad                       | C minor   |
| fingerprint_hash | hash         | huella digital del audio        | a9f12e    |


* Variables / Futures usadas en sonido:

| Feature            | Qué mide          |
| ------------------ | ----------------- |
| MFCC               | timbre del sonido |
| chroma             | notas musicales   |
| spectral centroid  | brillo del sonido |
| tempo              | BPM               |
| zero crossing rate | ruido             |


## Objetivos del proyecto



* Desarrollar un sistema de análisis de datos que permita comprender el comportamiento del mercado musical digital.

* Implementar modelos de machine learning para predecir el rendimiento de canciones y recomendar estrategias de monetización.

* Diseñar un sistema de detección de anomalías aplicado a datos de streaming para identificar posibles fraudes.

* Construir un prototipo de reconocimiento de audio para la identificación de contenido protegido por derechos de autor.

* Integrar estos componentes en una arquitectura escalable que pueda evolucionar hacia una plataforma real.

## Hipótesis de trabajo



* Es posible predecir el éxito relativo de una canción a partir de variables como sus características de audio, contexto de lanzamiento y métricas de engagement.

* Los patrones de fraude en streaming pueden ser identificados mediante comportamientos anómalos en los datos de escucha, sin necesidad de información sensible como direcciones IP.

* Las técnicas de procesamiento de audio permiten detectar coincidencias entre contenidos musicales con un alto grado de precisión, incluso en entornos ruidosos o transformados.

* El uso combinado de análisis de datos e inteligencia artificial puede ayudar a los artistas independientes a optimizar sus ingresos y tomar decisiones estratégicas más informadas.

# Data encontrada:


## **1. Analisis del mercado musical apra el artista:**

Utilizar los recursos de búsqueda y metadatos de Freesound:

. Permite extraer etiquetas (tags), categorías, nombres de sonidos y descripciones

. Además, ofrece métricas de "éxito" o popularidad como el número de descargas (num_downloads) y la calificación promedio (avg_rating)

. Esto te permitiría identificar qué géneros o tipos de sonidos tienen mayor demanda o mejores valoraciones

### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

### API Request 

In [ ]:
ret = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=chart.gettoptracks&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret.json()

In [ ]:
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")

In [ ]:
music = pd.DataFrame(ret.json()["tracks"]["track"])
music.head()

In [ ]:
# Cleaning the data frame:
music.drop(['mbid', 'image', 'url'],axis=1)

### Filtramos la información que queremos observar

In [ ]:
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")

In [ ]:
df_top30_duration.loc[:, 'duration'] = pd.to_numeric(df_top30_duration['duration'], errors='coerce')

mean = df_top30_duration['duration'].mean()
mean


### Duración promedio de top30 tracks:

In [ ]:
df_top30_duration = music.iloc[:30]
df_top30_duration

### Oyentes de los artistas top 10:

In [ ]:
df_top10 = music.head(10)
df_top10

In [ ]:
df_top10['duration'] = pd.to_numeric(df_top10['duration'], errors='coerce')
df_top10['listeners'] = pd.to_numeric(df_top10['listeners'], errors='coerce')

df_top10['artist_name'] = df_top10['artist'].apply(lambda x: x['name'])

plt.figure(figsize = (15, 5))

plt.bar(df_top10['artist_name'], df_top10['listeners'], color = ['skyblue'])
plt.xlabel('Artist name')
plt.ylabel('Listeners count')
plt.title("Listener of top 10 artist")
plt.show()

### Procedencia de los artisitas top10:

In [ ]:
# Info api request:
ret_artist = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=geo.gettopartists&country=spain&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret_artist.json()

In [ ]:
artists_country = pd.DataFrame(ret_artist.json()["topartists"]["artist"])
artists_country.head()

### Comparación de grupos (canciones 'happy'-'sad'):

In [ ]:
df_happy = music[music['name'].str.contains('happy',case=False,na=False)]
df_happy.count()

In [ ]:
df_sad = music[music['name'].str.contains('sad', case=False, na=False)]
df_sad.count()

## **2. Fraude en streams:**

## **3. Detector de copyright usando IA (Music Information Retrieval (MIR)):**

Freesound proporciona descriptores de audio precomputados:

. Puedes obtener datos técnicos como: 

* MFCC (Mel-frequency cepstrum coefficients): Esenciales para la identificación de timbres

* Vectores de similitud (como LAION-CLAP): Diseñados para capturar propiedades acústicas y semánticas, ideales para sistemas de huella digital (audio fingerprinting)

. Análisis espectral: Incluye centroide espectral, complejidad y energía, que sirven para crear los espectrogramas que mencionas

In [ ]:

# Para análisis de mercado
import pandas as pd
df_artist = pd.read_csv("https://freesound.org/docs/api/resources_apiv2.html#sound-resources", sep=",") 
df_artist.head()